In [ ]:
import os
import gc
import pickle

import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import VotingRegressor

import lightgbm as lgb
import catboost as cb

import optuna

import matplotlib.pyplot as plt



In [ ]:
class MonthlyKFold:
    def __init__(self, n_splits=3):
        self.n_splits = n_splits
        
    def split(self, X, y, groups=None):
        dates = 12 * X["year"] + X["month"]
        timesteps = sorted(dates.unique().tolist())
        X = X.reset_index()
        
        for t in timesteps[-self.n_splits:]:
            idx_train = X[dates.values < t].index
            idx_test = X[dates.values == t].index
            
            yield idx_train, idx_test
            
    def get_n_splits(self, X, y, groups=None):
        return self.n_splits

In [ ]:
def feature_eng(df_data, df_client, df_gas, df_electricity, df_forecast, df_historical, df_location, df_target):
    df_data = (
        df_data
        .with_columns(
            pl.col("datetime").cast(pl.Date).alias("date"),
        )
    )
    
    df_client = (
        df_client
        .with_columns(
            (pl.col("date") + pl.duration(days=2)).cast(pl.Date)
        )
    )
    
    df_gas = (
        df_gas
        .rename({"forecast_date": "date"})
        .with_columns(
            (pl.col("date") + pl.duration(days=1)).cast(pl.Date)
        )
    )
    
    df_electricity = (
        df_electricity
        .rename({"forecast_date": "datetime"})
        .with_columns(
            pl.col("datetime") + pl.duration(days=1)
        )
    )
    
    df_location = (
        df_location
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32)
        )
    )
    
    df_forecast = (
        df_forecast
        .rename({"forecast_datetime": "datetime"})
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32),
            pl.col('datetime').dt.replace_time_zone(None).cast(pl.Datetime("us")),
            #pl.col('datetime').dt.convert_time_zone("Europe/Bucharest").dt.replace_time_zone(None).cast(pl.Datetime("us")),
        )
        .join(df_location, how="left", on=["longitude", "latitude"])
        .drop("longitude", "latitude")
    )
    
    df_historical = (
        df_historical
        .with_columns(
            pl.col("latitude").cast(pl.datatypes.Float32),
            pl.col("longitude").cast(pl.datatypes.Float32),
            pl.col("datetime") + pl.duration(hours=37)
        )
        .join(df_location, how="left", on=["longitude", "latitude"])
        .drop("longitude", "latitude")
    )
    
    df_forecast_date = (
        df_forecast
        .group_by("datetime").mean()
        .drop("county")
    )
    
    df_forecast_local = (
        df_forecast
        .filter(pl.col("county").is_not_null())
        .group_by("county", "datetime").mean()
    )
    
    df_historical_date = (
        df_historical
        .group_by("datetime").mean()
        .drop("county")
    )
    
    df_historical_local = (
        df_historical
        .filter(pl.col("county").is_not_null())
        .group_by("county", "datetime").mean()
    )
    # Объединение всех обработанных данных с основным датафреймом df_data
    df_data = (
        df_data
        .join(df_gas, on="date", how="left")
        .join(df_client, on=["county", "is_business", "product_type", "date"], how="left")
        .join(df_electricity, on="datetime", how="left")
        
        .join(df_forecast_date, on="datetime", how="left", suffix="_fd")
        .join(df_forecast_local, on=["county", "datetime"], how="left", suffix="_fl")
        .join(df_historical_date, on="datetime", how="left", suffix="_hd")
        .join(df_historical_local, on=["county", "datetime"], how="left", suffix="_hl")
        
        .join(df_forecast_date.with_columns(pl.col("datetime") + pl.duration(days=7)), on="datetime", how="left", suffix="_fdw")
        .join(df_forecast_local.with_columns(pl.col("datetime") + pl.duration(days=7)), on=["county", "datetime"], how="left", suffix="_flw")
        .join(df_historical_date.with_columns(pl.col("datetime") + pl.duration(days=7)), on="datetime", how="left", suffix="_hdw")
        .join(df_historical_local.with_columns(pl.col("datetime") + pl.duration(days=7)), on=["county", "datetime"], how="left", suffix="_hlw")
        
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=2)).rename({"target": "target_1"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=3)).rename({"target": "target_2"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=4)).rename({"target": "target_3"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=5)).rename({"target": "target_4"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=6)).rename({"target": "target_5"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=7)).rename({"target": "target_6"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        .join(df_target.with_columns(pl.col("datetime") + pl.duration(days=14)).rename({"target": "target_7"}), on=["county", "is_business", "product_type", "is_consumption", "datetime"], how="left")
        # Создание категориальных признаков и тригонометрических функций времени
        .with_columns(
            pl.col("datetime").dt.ordinal_day().alias("dayofyear"), # Добавление номера дня в году
            pl.col("datetime").dt.hour().alias("hour"),# Добавление часа
            pl.col("datetime").dt.day().alias("day"),# Добавление дня
            pl.col("datetime").dt.weekday().alias("weekday"),# Добавление дня недели
            pl.col("datetime").dt.month().alias("month"),# Добавление месяца
            pl.col("datetime").dt.year().alias("year"),# Добавление года
        )
        # Приведение типов данных
        .with_columns(
            pl.concat_str("county", "is_business", "product_type", "is_consumption", separator="_").alias("category_1"),
        )
        
        .with_columns(
            (np.pi * pl.col("dayofyear") / 183).sin().alias("sin(dayofyear)"), # Тригонометрические функции для дня в году
            (np.pi * pl.col("dayofyear") / 183).cos().alias("cos(dayofyear)"),
            (np.pi * pl.col("hour") / 12).sin().alias("sin(hour)"),
            (np.pi * pl.col("hour") / 12).cos().alias("cos(hour)"),
        )
        
        .with_columns(
            pl.col(pl.Float64).cast(pl.Float32),
        )
         # Удаление ненужных колонок
        .drop("date", "datetime", "hour", "dayofyear")
    )
    
    return df_data

In [ ]:
def to_pandas(X, y=None):
    cat_cols = ["county", "is_business", "product_type", "is_consumption", "category_1"]
    
    if y is not None:
        df = pd.concat([X.to_pandas(), y.to_pandas()], axis=1)
    else:
        df = X.to_pandas()    
    
    df = df.set_index("row_id")
    df[cat_cols] = df[cat_cols].astype("category")
    
    df["target_mean"] = df[[f"target_{i}" for i in range(1, 7)]].mean(1)
    df["target_std"] = df[[f"target_{i}" for i in range(1, 7)]].std(1)
    df["target_ratio"] = df["target_6"] / (df["target_7"] + 1e-3)
    
    return df

In [ ]:
# для оптуны
def lgb_objective(trial):
    params = {
        'n_iter'           : 1000,
        'verbose'          : -1,
        'random_state'     : 42,
        'objective'        : 'l2',
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bynode' : trial.suggest_float('colsample_bynode', 0.5, 1.0),
        'lambda_l1'        : trial.suggest_float('lambda_l1', 1e-2, 10.0),
        'lambda_l2'        : trial.suggest_float('lambda_l2', 1e-2, 10.0),
        'min_data_in_leaf' : trial.suggest_int('min_data_in_leaf', 4, 256),
        'max_depth'        : trial.suggest_int('max_depth', 5, 10),
        'max_bin'          : trial.suggest_int('max_bin', 32, 1024),
    }
    
    model  = lgb.LGBMRegressor(**params)
    X, y   = df_train.drop(columns=["target"]), df_train["target"]
    cv     = MonthlyKFold(1)
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error')
    
    return -1 * np.mean(scores)

# Исследование

In [ ]:
# Загрузка данных об энергопотреблении
train = pd.read_csv("/kaggle/input/predict-energy-behavior-of-prosumers/train.csv")

# Создание сводной таблицы с средними значениями целевой переменной (target)
# для каждой комбинации даты, округа, типа продукта, бизнеса и потребления
pivot_train = train.pivot_table(
    index='datetime',
    columns=['county', 'product_type', 'is_business', 'is_consumption'],
    values='target',
    aggfunc='mean'
)

# Переименование колонок для удобства доступа и интерпретации
pivot_train.columns = ['county{}_productType{}_isBusiness{}_isConsumption{}'.format(*col) for col in pivot_train.columns.values]
pivot_train.index = pd.to_datetime(pivot_train.index)

pivot_train

## 2023 год 

In [ ]:
# Копирование сводной таблицы для визуализации
df_plot = pivot_train.copy()

# Нормализация данных для визуализации
df_plot = (df_plot - df_plot.min()) / (df_plot.max() - df_plot.min())

# Ресемплирование данных по дням и вычисление средних значений
df_plot_resampled_D = df_plot.resample('D').mean()

# Визуализация нормализованных данных с прозрачностью (alpha=0.1)
df_plot_resampled_D.loc['2022-7':].plot(alpha=0.1, color='green', figsize=(18, 6), legend=False)


In [ ]:
# Выбор колонок, соответствующих различным категориям потребления
columns_consumption_0 = df_plot_resampled_D.columns[df_plot_resampled_D.columns.str.contains('isConsumption0')]
columns_consumption_1 = df_plot_resampled_D.columns[df_plot_resampled_D.columns.str.contains('isConsumption1')]

# Создание фигуры для визуализации
plt.figure(figsize=(15, 6))

# Создание пустых линий для легенды
plt.plot([], color='red', label='is_Consumption = 1')  # Изменено на желтый цвет
plt.plot([], color='black', label='is_Consumption = 0')   # Изменено на черный цвет

# Отображение легенды
plt.legend()

# Визуализация данных для 'is_Consumption = 0' черным цветом
for column in columns_consumption_0:
    df_plot_resampled_D.loc['2022-7':, column].plot(alpha=0.1, color='black', legend=False)  # Изменено на черный

# Визуализация данных для 'is_Consumption = 1' желтым цветом
for column in columns_consumption_1:
    df_plot_resampled_D.loc['2022-7':, column].plot(alpha=0.1, color='red', legend=False)  # Изменено на желтый

# Отображение графика
plt.show()




### Global Variables

In [ ]:
root = "/kaggle/input/predict-energy-behavior-of-prosumers"

data_cols        = ['target', 'county', 'is_business', 'product_type', 'is_consumption', 'datetime', 'row_id']
client_cols      = ['product_type', 'county', 'eic_count', 'installed_capacity', 'is_business', 'date']
gas_cols         = ['forecast_date', 'lowest_price_per_mwh', 'highest_price_per_mwh']
electricity_cols = ['forecast_date', 'euros_per_mwh']
forecast_cols    = ['latitude', 'longitude', 'hours_ahead', 'temperature', 'dewpoint', 'cloudcover_high', 'cloudcover_low', 'cloudcover_mid', 'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'forecast_datetime', 'direct_solar_radiation', 'surface_solar_radiation_downwards', 'snowfall', 'total_precipitation']
historical_cols  = ['datetime', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure','cloudcover_total','cloudcover_low','cloudcover_mid','cloudcover_high','windspeed_10m','winddirection_10m','shortwave_radiation','direct_solar_radiation','diffuse_radiation','latitude','longitude']
location_cols    = ['longitude', 'latitude', 'county']
target_cols      = ['target', 'county', 'is_business', 'product_type', 'is_consumption', 'datetime']

save_path = None
load_path = None

### Data I/O

In [ ]:
df_data        = pl.read_csv(os.path.join(root, "train.csv"), columns=data_cols, try_parse_dates=True)
df_client      = pl.read_csv(os.path.join(root, "client.csv"), columns=client_cols, try_parse_dates=True)
df_gas         = pl.read_csv(os.path.join(root, "gas_prices.csv"), columns=gas_cols, try_parse_dates=True)
df_electricity = pl.read_csv(os.path.join(root, "electricity_prices.csv"), columns=electricity_cols, try_parse_dates=True)
df_forecast    = pl.read_csv(os.path.join(root, "forecast_weather.csv"), columns=forecast_cols, try_parse_dates=True)
df_historical  = pl.read_csv(os.path.join(root, "historical_weather.csv"), columns=historical_cols, try_parse_dates=True)
df_location    = pl.read_csv(os.path.join(root, "weather_station_to_county_mapping.csv"), columns=location_cols, try_parse_dates=True)
df_target      = df_data.select(target_cols)

schema_data        = df_data.schema
schema_client      = df_client.schema
schema_gas         = df_gas.schema
schema_electricity = df_electricity.schema
schema_forecast    = df_forecast.schema
schema_historical  = df_historical.schema
schema_target      = df_target.schema

### Feature Engineering

In [ ]:
X, y = df_data.drop("target"), df_data.select("target")

X = feature_eng(X, df_client, df_gas, df_electricity, df_forecast, df_historical, df_location, df_target)

df_train = to_pandas(X, y)

In [ ]:
df_train = df_train[df_train["target"].notnull() & df_train["year"].gt(2021)]

In [ ]:
# df_train.info(verbose=True)

### HyperParam Optimization

In [ ]:
# study = optuna.create_study(direction='minimize', study_name='Regressor')
# study.optimize(lgb_objective, n_trials=100, show_progress_bar=True)

### Validation

In [ ]:
'''result = cross_validate(
    estimator=lgb.LGBMRegressor(**best_params, random_state=42),
    X=df_train.drop(columns=["target"]), 
    y=df_train["target"],
    scoring="neg_mean_absolute_error",
    cv=MonthlyKFold(1),
)

print(f"Fit Time(s): {result['fit_time'].mean():.3f}")
print(f"Score Time(s): {result['score_time'].mean():.3f}")
print(f"Error(MAE): {-result['test_score'].mean():.3f}")'''

### Training

In [ ]:
# #LGB
p1={'n_estimators': 1000,'verbose': -1,'objective': 'l2','learning_rate': 0.06258413085998576, 'colsample_bytree': 0.6527661140701613, 'colsample_bynode': 0.8106858631408332, 'lambda_l1': 5.065645378814257, 'lambda_l2': 9.81159370218779, 'min_data_in_leaf': 192, 'max_depth': 10, 'max_bin': 1800}
p2={'n_estimators': 1000,'verbose': -1,'objective': 'l2','learning_rate': 0.0632167263149817, 'colsample_bytree': 0.6958033941948067, 'colsample_bynode': 0.6030801666196094, 'lambda_l1': 7.137580620471935, 'lambda_l2': 9.348169401713742, 'min_data_in_leaf': 74, 'max_depth': 11, 'max_bin': 530}
p3={'n_estimators': 1000,'verbose': -1,'objective': 'l2','learning_rate': 0.061236402165228264, 'colsample_bytree': 0.81427095118471, 'colsample_bynode': 0.6097376843527067, 'lambda_l1': 6.360490880385201, 'lambda_l2': 9.954136008333839, 'min_data_in_leaf': 238, 'max_depth': 13, 'max_bin': 649}
p4={'n_estimators': 1000,'verbose': -1,'objective': 'l2','learning_rate': 0.06753282378023663, 'colsample_bytree': 0.7508715107428325, 'colsample_bynode': 0.6831819500325418, 'lambda_l1': 8.679353563755722, 'lambda_l2': 6.105008696961338, 'min_data_in_leaf': 198, 'max_depth': 15, 'max_bin': 835}
p5={'n_estimators': 1000,'verbose': -1,'objective': 'l2','learning_rate': 0.05129380742257108, 'colsample_bytree': 0.5101576947777211, 'colsample_bynode': 0.8052639518604396, 'lambda_l1': 8.087311995794915, 'lambda_l2': 5.067361158677095, 'min_data_in_leaf': 222, 'max_depth': 8, 'max_bin': 97}
p6={'n_estimators': 900,'verbose': -1,'objective': 'l2','learning_rate': 0.05689066836106983,'colsample_bytree': 0.8915976762048253,'colsample_bynode': 0.5942203285139224,'lambda_l1': 3.6277555139102864,'lambda_l2': 1.6591278779517808,'min_data_in_leaf' : 186,'max_depth': 9,'max_bin': 813,}

In [ ]:
# #cat
# c1 = {'learning_rate': 0.06258413085998576, 'depth': 10, 'l2_leaf_reg': 8, 'border_count': 211, 'random_strength': 6, 'bagging_temperature': 0.13029094645654574}
# c2 = {'learning_rate': 0.08766355644863072, 'depth': 10, 'l2_leaf_reg': 8, 'border_count': 231, 'random_strength': 5, 'bagging_temperature': 0.20294125980762928}
# c3 = {'learning_rate': 0.14331003896351277, 'depth': 7, 'l2_leaf_reg': 10, 'border_count': 162, 'random_strength': 2, 'bagging_temperature': 0.3016878017499466}
# c4 = {'learning_rate': 0.13382144579754543, 'depth': 7, 'l2_leaf_reg': 10, 'border_count': 179, 'random_strength': 5, 'bagging_temperature': 0.30199258643156335}
# c5 = {'learning_rate': 0.12358952478027072, 'depth': 11, 'l2_leaf_reg': 8, 'border_count': 191, 'random_strength': 3, 'bagging_temperature': 0.41774414265586035}
# c6 = {'learning_rate': 0.13767418996955944, 'depth': 7, 'l2_leaf_reg': 10, 'border_count': 182, 'random_strength': 3, 'bagging_temperature': 0.42796507669281386}


500
1000
2000
3000


In [ ]:
c1 = {
    'learning_rate': 0.06258413085998576,
    'depth': 10,
    'l2_leaf_reg': 8,
    'border_count': 211,
    'random_strength': 6,
    'bagging_temperature': 0.13029094645654574,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

c2 = {
    'learning_rate': 0.08766355644863072,
    'depth': 10,
    'l2_leaf_reg': 8,
    'border_count': 231,
    'random_strength': 5,
    'bagging_temperature': 0.20294125980762928,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

c3 = {
    'learning_rate': 0.14331003896351277,
    'depth': 7,
    'l2_leaf_reg': 10,
    'border_count': 162,
    'random_strength': 2,
    'bagging_temperature': 0.3016878017499466,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

c4 = {
    'learning_rate': 0.13382144579754543,
    'depth': 7,
    'l2_leaf_reg': 10,
    'border_count': 179,
    'random_strength': 5,
    'bagging_temperature': 0.30199258643156335,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

c5 = {
    'learning_rate': 0.12358952478027072,
    'depth': 11,
    'l2_leaf_reg': 8,
    'border_count': 191,
    'random_strength': 3,
    'bagging_temperature': 0.41774414265586035,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

c6 = {
    'learning_rate': 0.13767418996955944,
    'depth': 7,
    'l2_leaf_reg': 10,
    'border_count': 182,
    'random_strength': 3,
    'bagging_temperature': 0.42796507669281386,
    'iterations': 15,
    'eval_metric': 'MAE',
    'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

}

In [ ]:
# c1 = {
#     'learning_rate': 0.06258413085998576,
#     'depth': 10,
#     'l2_leaf_reg': 8,
#     'border_count': 211,
#     'random_strength': 6,
#     'bagging_temperature': 0.13029094645654574,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

# c2 = {
#     'learning_rate': 0.08766355644863072,
#     'depth': 10,
#     'l2_leaf_reg': 8,
#     'border_count': 231,
#     'random_strength': 5,
#     'bagging_temperature': 0.20294125980762928,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

# c3 = {
#     'learning_rate': 0.14331003896351277,
#     'depth': 7,
#     'l2_leaf_reg': 10,
#     'border_count': 162,
#     'random_strength': 2,
#     'bagging_temperature': 0.3016878017499466,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

# c4 = {
#     'learning_rate': 0.13382144579754543,
#     'depth': 7,
#     'l2_leaf_reg': 10,
#     'border_count': 179,
#     'random_strength': 5,
#     'bagging_temperature': 0.30199258643156335,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

# c5 = {
#     'learning_rate': 0.12358952478027072,
#     'depth': 11,
#     'l2_leaf_reg': 8,
#     'border_count': 191,
#     'random_strength': 3,
#     'bagging_temperature': 0.41774414265586035,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

# c6 = {
#     'learning_rate': 0.13767418996955944,
#     'depth': 7,
#     'l2_leaf_reg': 10,
#     'border_count': 182,
#     'random_strength': 3,
#     'bagging_temperature': 0.42796507669281386,
#     'iterations': 1500,
#     'eval_metric': 'MAE',
#     'cat_features': ['county', 'is_business', 'product_type', 'is_consumption', 'category_1'],

# }

## NEW

In [ ]:
# # солнечная энергия 

# if load_path is not None:
#     model = pickle.load(open(load_path, "rb"))
# else:
#     model = VotingRegressor([
#         ('catboost_1', cb.CatBoostRegressor(**c1, random_state=42)),
#         ('catboost_2', cb.CatBoostRegressor(**c2, random_state=42)),
#         ('catboost_3', cb.CatBoostRegressor(**c3, random_state=42)),
#         ('catboost_4', cb.CatBoostRegressor(**c4, random_state=42)),
#         ('catboost_5', cb.CatBoostRegressor(**c5, random_state=42))
#     ])
      
#     model_solar = VotingRegressor([
#         ('catboost_1', cb.CatBoostRegressor(**c1, random_state=42)),
#         ('catboost_2', cb.CatBoostRegressor(**c2, random_state=42)),
#         ('catboost_3', cb.CatBoostRegressor(**c3, random_state=42)),
#         ('catboost_4', cb.CatBoostRegressor(**c4, random_state=42)),
#         ('catboost_5', cb.CatBoostRegressor(**c5, random_state=42))
#     ])
    
    
    
#     model.fit(
#         X=df_train.drop(columns=["target"]),
#         y=df_train["target"]
#     )
    
#     model_solar.fit(
#         X=df_train[df_train['is_consumption']==0].drop(columns=["target"]),
#         y=df_train[df_train['is_consumption']==0]["target"]
#     )

# if save_path is not None:
#     with open(save_path, "wb") as f:
#         pickle.dump(model, f)
#     with open(save_path, "wb") as f:
#         pickle.dump(model_solar, f)


In [ ]:
# солнечная энергия 

if load_path is not None:
    model = pickle.load(open(load_path, "rb"))
else:
    model = VotingRegressor([
        ('lgb_1', lgb.LGBMRegressor(**p1, random_state=42)), 
        ('lgb_2', lgb.LGBMRegressor(**p2, random_state=42)), 
        ('lgb_3', lgb.LGBMRegressor(**p3, random_state=42)), 
        ('lgb_4', lgb.LGBMRegressor(**p4, random_state=42)), 
        ('lgb_5', lgb.LGBMRegressor(**p5, random_state=42)), 
    ])
     
        
    # Обучение каждой модели CatBoost отдельно
    models = [cb.CatBoostRegressor(**c1, random_state=42),
              cb.CatBoostRegressor(**c5, random_state=42), 
              cb.CatBoostRegressor(**c1, random_state=42),
              cb.CatBoostRegressor(**c4, random_state=42), 
              cb.CatBoostRegressor(**c5, random_state=42),
              cb.CatBoostRegressor(**c1, random_state=42)]

    for i, model in enumerate(models):
        model.fit(
            X=df_train[df_train['is_consumption']==0].drop(columns=["target"]),
            y=df_train[df_train['is_consumption']==0]["target"]
        )
        model.save_model(f'/kaggle/working/model_solar_part_{i}.cbm')
        print(f'запись model_solar_part{i}_.cbm окончена')
# !! ИНИЦИАЛИЗАЦИЯ ПЕРЕД ЗАГРУЗКОЙ НУЖНА!!!
    # Загрузка моделей
#     models = []
#     for i in range(6):
#         model = cb.CatBoostRegressor()
#         model.load_model(f'model_solar_part_{i}.cbm')
#         models.append(model)
#     print('загрузка окончена')
    #После обучения и сохранения моделей, создание VotingRegressor с уже обученными моделями
    
    
    
    model_solar = VotingRegressor([
        ('catboost_1', models[0]),
        ('catboost_2', models[1]),
        ('catboost_3', models[2]),
        ('catboost_4', models[3]),
        ('catboost_5', models[4]),
        ('catboost_6', models[5])
    ])    
        
    model_solar.fit(
       X=df_train[df_train['is_consumption']==0].drop(columns=["target"]),
       y=df_train[df_train['is_consumption']==0]["target"]
)
     
#     model_solar = VotingRegressor([
#         ('catboost_1', cb.CatBoostRegressor(**c1, random_state=42)),
#         ('catboost_2', cb.CatBoostRegressor(**c5, random_state=42)),
#         ('catboost_3', cb.CatBoostRegressor(**c1, random_state=42)),
#         ('catboost_4', cb.CatBoostRegressor(**c4, random_state=42)),
#         ('catboost_5', cb.CatBoostRegressor(**c5, random_state=42)),
#         ('catboost_6', cb.CatBoostRegressor(**c1, random_state=42))
#     ])
    
    
    
    model.fit(
        X=df_train.drop(columns=["target"]),
        y=df_train["target"]
    )
    
#     model_solar.fit(
#         X=df_train[df_train['is_consumption']==0].drop(columns=["target"]),
#         y=df_train[df_train['is_consumption']==0]["target"]
#     )
#     # Сохранение каждой модели в VotingRegressor отдельно
#     for i, (name, model) in enumerate(model_solar.named_estimators.items()):
#         model.save_model(f'/kaggle/working/model/model_solar_part_{i}.cbm')

    
if save_path is not None:
    with open(save_path, "wb") as f:
        pickle.dump(model, f)
    with open(save_path, "wb") as f:
        pickle.dump(model_solar, f)


In [ ]:
model_solar

In [ ]:
!tar -czf model.tar.gz -C /kaggle/working .

### Prediction

In [ ]:
import enefit

env = enefit.make_env()
iter_test = env.iter_test()

In [ ]:
# sample_prediction

In [ ]:
# # Подробные комментарии к коду на русском языке

# for (test, 
#      revealed_targets_test, 
#      client_test,
#      historical_weather_test,
#      forecast_weather_test,
#      electricity_test,
#      gas_test,
#      sample_prediction) in iter_test:
    
#     # Переименовываем колонки в тестовом наборе данных для согласованности с тренировочным набором
#     test = test.rename(columns = {'prediction_datetime': 'datetime'})
#     test['target'] = 0
#     first_datetime = test['datetime'].iloc[0]
    
#     # Подготовка данных
#     (test, revealed_targets_test, client_test, historical_weather_mean_test,
#      forecast_weather_mean_test, electricity_test, gas_test) = prepairing_data(test, revealed_targets_test, client_test,
#                                                                                historical_weather_test, forecast_weather_test,
#                                                                                electricity_test, gas_test, weather_map)
    
#     # Слияние данных
#     df_test = merge_data(test, revealed_targets_test, client_test,
#                          historical_weather_mean_test, forecast_weather_mean_test,
#                          electricity_test, gas_test, working_days)
    
#     # Создание признаков
#     df_test = features_maker(df_test)

#     # Переформатирование df_test
#     resampled_test = resample_data(df_test)
    
#     # Определение колонок признаков и целевой переменной
#     remove_columns = ['datetime', 'target', 'date', 'group']
#     features_1d = [f for f in resampled_test.columns if f not in remove_columns]
#     features = [f for f in df_test.columns if f not in remove_columns]
#     target = ['target']
    
#     # Предсказания для 1-дневного прогноза
#     resampled_test['lin_predict_1d'] = lin_reg_1d.predict(resampled_test[features_1d])
#     resampled_test['lin_predict_1d'] = resampled_test['lin_predict_1d'].clip(lower=0)
#     features_1d.append('lin_predict_1d')
    
#     predictions_1d_test = resampled_test[['datetime', 'county', 'is_business', 'product_type', 'is_consumption', 'prediction_unit_id']]
#     predictions_1d_test['prediction_1d'] = model_1d.predict(resampled_test[features_1d]).astype(float)
#     predictions_1d_test['prediction_1d'] = predictions_1d_test['prediction_1d'].clip(lower=0)
#     predictions_1d_test = predictions_1d_test.rename(columns={'datetime': 'date'})
    
#     df_test = df_test.merge(predictions_1d_test, how='left',
#                             on=['date', 'county', 'is_business', 'product_type', 'is_consumption', 'prediction_unit_id'])
#     features.append('prediction_1d')
    
#     # Предсказания для 1-часового прогноза
#     df_test['lin_predict'] = lin_reg_1h.predict(df_test[features])
#     df_test['lin_predict'] = df_test['lin_predict'].clip(0)
#     features.append('lin_predict')
#     df_test['target'] = model_1h.predict(df_test[features]).astype(float)
#     df_test['target'] = df_test['target'].clip(0)
    
#     # Сортировка значений
#     df_test = df_test.merge(
#         test[['county', 'is_business', 'product_type', 'is_consumption', 'datetime', 'row_id', 'prediction_unit_id']],
#         how='right', on=['county', 'is_business', 'product_type', 'is_consumption', 'datetime', 'prediction_unit_id']
#     ).sort_values(by='row_id')
    
#     # Запись предсказаний в sample_prediction
#     sample_prediction["target"] = df_test['target']
    
#     # Отправка предсказаний
#     env.predict(sample_prediction)


In [ ]:
# Основной цикл для обработки данных тестового набора
for (test, revealed_targets, client, historical_weather,
        forecast_weather, electricity_prices, gas_prices, sample_prediction) in iter_test:
    
    # Переименование столбца для удобства
    test = test.rename(columns={"prediction_datetime": "datetime"})
    
    # Преобразование данных в формат Polars для дальнейшей обработки
    df_test           = pl.from_pandas(test[data_cols[1:]], schema_overrides=schema_data)
    df_client         = pl.from_pandas(client[client_cols], schema_overrides=schema_client)
    df_gas            = pl.from_pandas(gas_prices[gas_cols], schema_overrides=schema_gas)
    df_electricity    = pl.from_pandas(electricity_prices[electricity_cols], schema_overrides=schema_electricity)
    df_new_forecast   = pl.from_pandas(forecast_weather[forecast_cols], schema_overrides=schema_forecast)
    df_new_historical = pl.from_pandas(historical_weather[historical_cols], schema_overrides=schema_historical)
    df_new_target     = pl.from_pandas(revealed_targets[target_cols], schema_overrides=schema_target)
    
    # Объединение новых данных с существующими и удаление дубликатов
    df_forecast       = pl.concat([df_forecast, df_new_forecast]).unique()
    df_historical     = pl.concat([df_historical, df_new_historical]).unique()
    df_target         = pl.concat([df_target, df_new_target]).unique()
    
    # Применение функции инженерии признаков и преобразование данных обратно в pandas
    X_test = feature_eng(df_test, df_client, df_gas, df_electricity, df_forecast, df_historical, df_location, df_target)
    X_test = to_pandas(X_test)
    
    # Прогнозирование с использованием модели и ограничение предсказаний нулем
    test['target'] = model.predict(X_test).clip(0)
    test['target_solar'] = model_solar.predict(X_test).clip(0)
    # Замена прогнозов для непотребляющих клиентов на прогнозы солнечной энергии
    test.loc[test['is_consumption']==0, "target"] = test.loc[test['is_consumption']==0, "target_solar"]    
    
    # Обновление целевых значений в примере предсказания
    sample_prediction["target"] = test['target']
    
    # Отправка предсказаний в среду выполнения
    env.predict(sample_prediction)


In [ ]:
test

In [ ]:
  sample_prediction